# 01 Data Loading

This notebook validates the environment, discovers supported raw datasets, loads available files, and saves dataset inventory outputs for the current project phase.


In [ ]:
from __future__ import annotations

import platform
import sys
from pathlib import Path

def detect_project_root(start: Path | None = None) -> Path:
    start_path = (start or Path.cwd()).resolve()
    for candidate in (start_path, *start_path.parents):
        if (candidate / "src").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not detect the project root.")

PROJECT_ROOT = detect_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"Python executable: {sys.executable}")
print(f"Python version: {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")


In [ ]:
import pandas as pd
from IPython.display import display

from src.config import DATASET_METADATA_PATH, RAW_DATA_DIR, REPORTS_DIR, SUPPORTED_DATASET_EXTENSIONS
from src.data_loader import (
    build_dataset_inventory,
    discover_dataset_files,
    generate_dataset_summary,
    load_dataset,
    load_dataset_metadata,
    resolve_dataset_metadata,
)

print(f"Raw data directory: {RAW_DATA_DIR}")
print(f"Reports directory: {REPORTS_DIR}")
print(f"Supported extensions: {SUPPORTED_DATASET_EXTENSIONS}")
print(f"Dataset metadata file: {DATASET_METADATA_PATH}")


In [ ]:
dataset_paths = discover_dataset_files(RAW_DATA_DIR)
print(f"Discovered {len(dataset_paths)} supported raw dataset(s).")
for dataset_path in dataset_paths:
    print(f" - {dataset_path.relative_to(PROJECT_ROOT)}")


In [ ]:
if not dataset_paths:
    print("WARNING: No supported datasets were found under data/raw/. Add datasets before running later phases.")
else:
    metadata_map = load_dataset_metadata(DATASET_METADATA_PATH)
    preview_rows = []
    for dataset_path in dataset_paths:
        metadata = resolve_dataset_metadata(dataset_path, metadata_map, raw_data_dir=RAW_DATA_DIR)
        dataframe = load_dataset(dataset_path)
        summary = generate_dataset_summary(dataframe, dataset_path, raw_data_dir=RAW_DATA_DIR, metadata=metadata)
        preview_rows.append({
            "dataset": dataset_path.name,
            "rows": summary["row_count"],
            "columns": summary["column_count"],
            "candidate_targets": ", ".join(summary["candidate_target_columns"]),
            "unit_of_analysis_note": summary["unit_of_analysis_note"],
        })
    display(pd.DataFrame(preview_rows))


In [ ]:
inventory_df, quality_df = build_dataset_inventory(
    raw_data_dir=RAW_DATA_DIR,
    metadata_path=DATASET_METADATA_PATH,
    output_dir=REPORTS_DIR,
)

print("Inventory reports written to:")
print(f" - {REPORTS_DIR / "dataset_inventory.csv"}")
print(f" - {REPORTS_DIR / "dataset_inventory.json"}")
print(f" - {REPORTS_DIR / "data_quality_summary.csv"}")

display(inventory_df.head())
display(quality_df.head())
